# Basic application

This notebook shows you how to use the *basic* Vitis network example that can be generated from this repository.
The design provides network connectivity using User Datagram Protocol (UDP) as the transport protocol.

The assumptions made is this notebook are:
1. You have an Alveo card configured with one of the supported shells
1. You have generated the xclbin file for the *basic* example
1. You have a 100 GbE capable NIC
1. Both the Alveo card and the 100 GbE capable NIC are in the same host
1. Alveo card is connected to the NIC, either directly or with a any network equipment, using any of the interfaces.

Let's have a look at the *basic* design.

There are 4 Kernels:
* CMAC: provides the translation between physical signals to AXI4-Stream interface
* Network layer: provides a bridge between raw Ethernet packets and the application using UDP as transport layer
    * ARP provides translation between MAC and IP addresses
    * ICMP provides ping capabilities
    * The UDP module has a 16-entry table with socket information that needs to be filled in before running
* krnl_mm2s: reads data from memory and packetize it setting tdest and tlast appropriately
* krnl_s2mm: read data from the stream and copy it to memory

![](../img/udp_network_basic.png)

**Note (project build)**: xclbin path updated to project matcher design (krnl_proj + krnl_s2mm). Ensure you set s2mm size to output bytes (input bytes * 2) and provide packets on nl2sk.


In the followings cells we will:
* Import the necessary pynq and python packages, define current device and xclbin file
* Explore kernels in the design
* Check physical link
* Change Alveo card IP address and ping it
* Configure socket table and populate it to the UDP module
* Create UDP socket in the host
* Allocate Alveo buffers
* Move data from the HOST through the network (NIC) to the Alveo card
* Move data from the HOST through the Alveo card to the network (NIC)
* Free resources

In [1]:
#Uncomment the next line and run to reset the FPGA if it is not taking programming or otherwise misbehaving
!xbutil reset --device 0000:02:00.1 --force

Performing 'HOT Reset' on '0000:02:00.1'
Are you sure you wish to proceed? [Y/n]: Y (Force override)
Successfully reset Device[0000:02:00.1]


## Import packages and program FPGA
In this section we need to import the `pynq` and python packages that will be used in the rest of this notebook. We also import the `vnx_utils.py` file with helper functions to set up the vnx examples.

In [2]:
import pynq
import numpy as np
from _thread import *
import threading 
import socket
from vnx_utils import *

We also need to define the current device, only if there is more than one Alveo card on the host. First let's check how many devices are available.

In [3]:
for i in range(len(pynq.Device.devices)):
    print("{}) {}".format(i, pynq.Device.devices[i].name))

0) xilinx_u55c_gen3x16_xdma_base_3


* If there are more than one Alveo card available, you should pass the `device` argument to the `pynq.Overlay` class.
* The xclbin variable should point to the `xclbin` file for the *basic* example

In [4]:
currentDevice = pynq.Device.devices[0]
xclbin = '/home/m2_1/dat480_project_base/project.intf0.xilinx_u55c_gen3x16_xdma_3_202210_1/vnx_project_if0.xclbin'
ol = pynq.Overlay(xclbin,device=currentDevice)

## Explore kernels in the design
This design was built for both interfaces. Therefore, we will see each kernel repeated twice. One for the interface 0 and another for the interface 1

In [5]:
ol.ip_dict

{'cmac_0': {'phys_addr': 12582912,
  'addr_range': 8192,
  'type': 'xilinx.com:kernel:cmac_0:1.0',
  'hw_control_protocol': 'ap_ctrl_none',
  'fullpath': 'cmac_0',
  'registers': {'gt_reset_reg': {'address_offset': 0,
    'access': 'read-write;',
    'size': 32,
    'host_size': 4,
    'description': 'OpenCL Argument Register',
    'type': 'uint',
    'id': 0},
   'reset_reg': {'address_offset': 4,
    'access': 'read-write;',
    'size': 32,
    'host_size': 4,
    'description': 'OpenCL Argument Register',
    'type': 'uint',
    'id': 1},
   'mode': {'address_offset': 8,
    'access': 'read-write;',
    'size': 32,
    'host_size': 4,
    'description': 'OpenCL Argument Register',
    'type': 'uint',
    'id': 2},
   'conf_tx': {'address_offset': 12,
    'access': 'read-write;',
    'size': 32,
    'host_size': 4,
    'description': 'OpenCL Argument Register',
    'type': 'uint',
    'id': 3},
   'conf_rx': {'address_offset': 20,
    'access': 'read-write;',
    'size': 32,
    'hos

## Check physical link
After the dynamic region of the Alveo card is programmed with the basic example we can start interacting with the design.

Let's check if the Alveo card has detected link with the network equipment. To do so, we will use one of the helper functions `link_status`

In [6]:
print("Link interface 0 {}".format(ol.cmac_0.link_status()))

Link interface 0 {'cmac_link': True}


## Change Alveo card IP address and ping it
By defaul the Alveo IP address is `192.168.0.5` and MAC address is `00:0A:35:02:9D:E5`. Let's change it to `192.168.100.2`, in this example we are using interface 0

After the IP address is changed, we can ping the Alveo card. The first attempts will fail but the remaining should work.

**Make sure you have configured the IP address of the 100 GbE capable NIC to be in the same subnetwork as the Alveo card**

In [7]:
alveo_ipaddr = '192.168.100.2'
print(ol.networklayer_0.set_ip_address(alveo_ipaddr, debug=True))

{'HWaddr': '00:0a:35:02:9d:02', 'inet addr': '192.168.100.2', 'gateway addr': '192.168.100.1', 'Mask': '255.255.255.0'}


* Check 100 GbE capable NIC configuration

In [8]:
!ip addr show ens1f0np0

3: ens1f0np0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    link/ether 0c:42:a1:ef:fd:5e brd ff:ff:ff:ff:ff:ff
    altname enp1s0f0np0
    inet 192.168.100.1/30 scope global ens1f0np0
       valid_lft forever preferred_lft forever


* Run ping to get ARP set up
Some of the attempts will fail but the remaining should work.

In [9]:
# Warm up ARP/ND first (early packets may drop), then check ping
!ping -c 2 -i 0.2 -W 1 $alveo_ipaddr -I ens1f0np0 || true
!ping -c 5 $alveo_ipaddr -I ens1f0np0

PING 192.168.100.2 (192.168.100.2) from 192.168.100.1 ens1f0np0: 56(84) bytes of data.
64 bytes from 192.168.100.2: icmp_seq=2 ttl=128 time=0.143 ms
64 bytes from 192.168.100.2: icmp_seq=3 ttl=128 time=0.134 ms
64 bytes from 192.168.100.2: icmp_seq=4 ttl=128 time=0.126 ms
64 bytes from 192.168.100.2: icmp_seq=5 ttl=128 time=0.158 ms

--- 192.168.100.2 ping statistics ---
5 packets transmitted, 4 received, 20% packet loss, time 4097ms
rtt min/avg/max/mdev = 0.126/0.140/0.158/0.011 ms


## Configure socket table and populate it to the UDP module

In this section we will configure the socket table in software and populate it to the UDP module in the Alveo card. 
1. Define a couple of connections 
1. The socket table is populated to the UDP module in the Alveo card using the `.populate_socket_table()` helper function

In [10]:
sw_ip = '192.168.100.1'
ol.networklayer_0.sockets[0] = (sw_ip, 50446, 60133, True)
ol.networklayer_0.sockets[1] = (sw_ip, 38746, 62781, True)

ol.networklayer_0.populate_socket_table(debug=True)

{'Number of Sockets': 16,
 'socket': {0: {'theirIP': '192.168.100.1',
   'theirPort': 50446,
   'myPort': 60133},
  1: {'theirIP': '192.168.100.1', 'theirPort': 38746, 'myPort': 62781}}}

## Create UDP socket in the host
In this part we will open an UDP socket in the host (software) to be able to communicate with the Alveo card through the network.

We will use the python socket API to do so, the socket will be binded to the port `38746`

In [11]:
SW_PORT = ol.networklayer_0.sockets[1]['theirPort']
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM) # UDP
sock.bind(('', SW_PORT))

## Allocate Alveo buffers

We need to allocate the buffers on the global memory of the Alveo card to be able to pull and push data from them. We also will initialize the the sending buffer, `mm2s_buf`, with random data. This random data will be sent later to the network.

1. Define alias for the application kernlels (`mm2s` and `s2mm`)
1. Define size and shape of the buffers
1. Allocate the buffers
1. Initialize sending buffer with random data

In [12]:
mm2s = ol.krnl_mm2s_0
s2mm = ol.krnl_s2mm_0
proj = ol.krnl_proj_0

# Buffers for TX (mm2s -> network)
size = 1408 * 100
shape = (size,1)

# RX settings for matcher path (nl2sk -> krnl_proj -> s2mm)
PKT_BYTES = 1408
NUM_PKTS = 1
rx_in_bytes = PKT_BYTES * NUM_PKTS
rx_out_bytes = rx_in_bytes * 2  # krnl_proj emits 2 beats per 64B input beat

if hasattr(ol, 'HBM0'):
    mm2s_buf = pynq.allocate(shape, dtype=np.uint8, target=ol.HBM0)
    proj_out_buf = pynq.allocate(rx_out_bytes, dtype=np.uint8, target=ol.HBM0)
else:
    mm2s_buf = pynq.allocate(shape, dtype=np.uint8, target=ol.bank1)
    proj_out_buf = pynq.allocate(rx_out_bytes, dtype=np.uint8, target=ol.bank1)


## Move data from the HOST through the network (NIC) to the Alveo card

In this section we will write data to the socket, which will send such data from the host to the Alveo card using the network.

Start streaming to memory mapped kernel, we need to specify how much data to expect from the network.

In [13]:
# Start matcher receive path: prepare output buffer, launch s2mm and krnl_proj
proj_out_buf[:] = 0
proj_out_buf.sync_to_device()
s2mm_wh = s2mm.start(proj_out_buf, rx_out_bytes)
proj_wh = proj.start(0, NUM_PKTS)


Initialize a buffer with random data, define the packet size and compute how many packets we need to send to transmit the whole buffer to the network. Write the data from the buffer into the socket in `BYTES_PER_PACKET` chunks.

In [14]:
# Deterministic payload: zeros + embed a known pattern (ID=1 from patterns.h)
udp_message_global = np.zeros(rx_in_bytes, dtype=np.uint8)
pattern = np.array([0x2F,0x62,0x6E,0x62,0x66,0x6F,0x72,0x6D,0x2E,0x63,0x67,0x69], dtype=np.uint8)
offsets = [128]  # byte offsets within the packet to plant the pattern
for off in offsets:
    udp_message_global[off:off+len(pattern)] = pattern

BYTES_PER_PACKET = PKT_BYTES
num_pkts = NUM_PKTS
alveo_port = ol.networklayer_0.sockets[1]['myPort']
for m in range(num_pkts):
    start = m * BYTES_PER_PACKET
    udp_message_local = udp_message_global[start : start + BYTES_PER_PACKET]
    sock.sendto(udp_message_local, (alveo_ipaddr, alveo_port))


* Wait for the s2mm kernel to receive all the data
* Move data from global memory to HOST memory
* Compare what was sent against what was received and print out result

In [15]:
proj_wh.wait()
s2mm_wh.wait()
proj_out_buf.sync_from_device()
out_u16 = np.frombuffer(proj_out_buf, dtype=np.uint16)
nz = np.nonzero(out_u16)[0]
print('Output nonzero count:', len(nz))
# Decode matches: each input beat -> two output beats, 32 IDs each
matches = []
for idx in nz:
    pid = int(out_u16[idx])
    beat_out = idx // 32
    pos = idx % 32
    byte_idx = pos + (0 if (beat_out % 2)==0 else 32)
    input_beat = beat_out // 2
    matches.append((input_beat, byte_idx, pid))
print('Matches (show up to 16):', matches[:16])
# Expected positions for the planted pattern (last byte of pattern determines match beat/byte)
last_byte_offsets = [off + len(pattern) - 1 for off in offsets]
expected = [((o)//64, (o)%64, 1) for o in last_byte_offsets]
print('Expected matches:', expected)
# No direct payload compare here because krnl_proj outputs IDs, not payload


Host sending data through the network and the host getting data from kernel was a: SUCCESS!. Total data transmitted 140,800 bytes to ('192.168.100.2', 62781)


## Move data from the HOST through the Alveo card to the network (NIC)

We need to create a new thread to read data from the socket, this mainly because there will be multiple packets. This is done in the `socket_receive_threaded` function

In [16]:
print_lock = threading.Lock()
done = threading.Event()
# thread function
def socket_receive_threaded(sock, size):
    BYTES_PER_PACKET = 1*1408
    shape_global = (size,1)
    shape_local = (BYTES_PER_PACKET,1)
    recv_data_global = np.empty(shape_global, dtype = np.uint8)
    data_partial = np.empty(shape_local, dtype = np.uint8)
    num_it = (size // BYTES_PER_PACKET)
    global mm2s_buf
    sum_bytes = 0
    connection = 'None'
    for m in range(num_it):
        res = sock.recvfrom_into(data_partial)
        recv_data_global[(m * BYTES_PER_PACKET) : ((m * BYTES_PER_PACKET) \
                        + BYTES_PER_PACKET)] = data_partial
        sum_bytes = sum_bytes + int(res[0])
        connection = res[1]
    msg = "SUCCESS!" if np.array_equal(mm2s_buf, recv_data_global) else f"FAILURE! {recv_data_global[:20]} {mm2s_buf[:20]}"
    print ("Kernel sending data to the network and the host getting data from network"
    " was a: {}. Total data received {:,} bytes from {}".format(msg,sum_bytes,connection))
    print_lock.release()
    done.set()

* Copy random data to Alveo global memory
* Acquire thread 
* Launch threaded function
* Start kernel memory mapped to stream kernel, this will start sending UDP packets from the Alveo card to the NIC
* Threaded function will start receiving data and it will print out result once ALL data was collected

In [17]:

mm2s_buf[:] = np.random.randint(low=0, high=((2**8)-1), size=shape, dtype=np.uint8)
mm2s_buf.sync_to_device()
print_lock.acquire()
start_new_thread(socket_receive_threaded, (sock,size,))
mm2s_wh = mm2s.start(mm2s_buf,size, 1)
done.wait()
done.clear()

Kernel sending data to the network and the host getting data from network was a: SUCCESS!. Total data received 140,800 bytes from ('192.168.100.2', 62781)


## Free resources
Delete buffers and free Alveo card

In [18]:
del mm2s_buf
del proj_out_buf
del udp_message_global
pynq.Overlay.free(ol)


In [19]:
# it is technically not "needed" to rest the FPGA when you are done with it, but it uses less power in reset state, so it's nice to do
!xbutil reset --device 0000:02:00.1 --force

Performing 'HOT Reset' on '0000:02:00.1'
Are you sure you wish to proceed? [Y/n]: Y (Force override)


: 

------------------------------------------
Copyright (c) 2020-2021, Xilinx, Inc.